# Notebook guiado — Tu primera Red Neuronal con Keras y PyTorch

**RA2 · Programación de Inteligencia Artificial**  
**Notebook de refuerzo · Google Colab**

En este notebook construiremos una red neuronal desde cero **entendiendo qué ocurre en cada paso**.

No empezaremos directamente con `model.fit()`.

Primero veremos:

**dato → neurona → suma ponderada → activación → capa → red → predicción → error → actualización de pesos**

Después entrenaremos la misma idea de red neuronal en dos frameworks:

1. **Keras** → API de alto nivel, compacta y muy cómoda para empezar.
2. **PyTorch** → entrenamiento más explícito, ideal para ver el ciclo completo.

## ¿Por qué no JAX aquí?

JAX es muy potente, pero introduce conceptos adicionales —programación funcional, transformaciones, `jit`, gestión explícita de parámetros, etc.— que no son necesarios para comprender una **primera red neuronal**.

Aquí queremos que quede muy claro:

- qué es una neurona;
- qué son pesos y bias;
- qué hace una activación;
- qué es una capa;
- qué significa forward pass;
- qué mide la loss;
- qué hace backpropagation;
- qué es un optimizador;
- qué significan batch, epoch y learning rate;
- cómo detectar sobreajuste;
- cómo realizar inferencia.

## Recorrido

1. De una neurona a una red
2. Pesos y bias
3. Funciones de activación
4. Forward pass
5. Por qué necesitamos no linealidad
6. Loss
7. Gradientes y backpropagation
8. Optimizadores
9. Epoch, batch y learning rate
10. Train / validation / test
11. Dataset visual
12. Red con Keras
13. Curvas de aprendizaje
14. Frontera de decisión
15. Inferencia
16. Red equivalente con PyTorch
17. `Dataset` y `DataLoader`
18. Training loop
19. Validación
20. Comparación Keras vs. PyTorch
21. Overfitting
22. Regularización
23. Learning rate
24. Batch size
25. Matriz de confusión
26. Probabilidades y logits
27. Qué ocurre dentro de una predicción
28. Mini reto final

> El objetivo es **entender una red neuronal**, no memorizar dos librerías.


# 0. Preparar el entorno

Google Colab incluye habitualmente TensorFlow/Keras y PyTorch.

Comprobaremos versiones y fijaremos semillas para que los resultados sean lo más reproducibles posible.


In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    ConfusionMatrixDisplay,
    classification_report
)

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
torch.manual_seed(SEED)

print("TensorFlow:", tf.__version__)
print("PyTorch:", torch.__version__)


## CPU o GPU

Para esta red tan pequeña, CPU es suficiente.

Aun así, comprobaremos si PyTorch detecta GPU.


In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Dispositivo PyTorch:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


# 1. ¿Qué es una neurona artificial?

Una neurona recibe varias características:

\[
x_1,\;x_2,\;...,\;x_n
\]

Cada entrada tiene asociado un peso:

\[
w_1,\;w_2,\;...,\;w_n
\]

Primero calcula:

\[
z =
w_1x_1 +
w_2x_2 +
... +
w_nx_n +
b
\]

donde \(b\) es el **bias**.

Después aplica una función de activación:

\[
a=f(z)
\]

Por tanto:

**entradas → pesos → suma → bias → activación → salida**


## ¿Qué representan los pesos?

Los pesos controlan cuánto influye cada entrada.

- peso positivo grande → favorece aumentar la salida;
- peso negativo → puede reducirla;
- peso cercano a cero → poca influencia directa.

Durante el entrenamiento, la red modifica estos pesos.

## ¿Y el bias?

Permite desplazar la función de decisión.

También es un parámetro aprendido.


# 2. Una neurona calculada manualmente

Supongamos:

\[
x_1=0.8,\quad x_2=0.3
\]

\[
w_1=1.2,\quad w_2=-0.7
\]

\[
b=0.1
\]

Calcularemos la preactivación \(z\).


In [ ]:
x = np.array([0.8, 0.3])
w = np.array([1.2, -0.7])
b = 0.1

z = np.dot(x, w) + b

print("Entradas:", x)
print("Pesos:", w)
print("Bias:", b)
print("z =", z)


# 3. Funciones de activación

Si una red solo encadenara transformaciones lineales, varias capas lineales podrían reducirse a otra transformación lineal.

Para aprender relaciones no lineales necesitamos activaciones.

## ReLU

\[
ReLU(z)=\max(0,z)
\]

## Sigmoid

\[
\sigma(z)=\frac{1}{1+e^{-z}}
\]

ReLU es muy habitual en capas ocultas.

Sigmoid puede utilizarse, entre otros contextos, para convertir una única salida en una probabilidad binaria.


In [ ]:
z_values = np.linspace(-6, 6, 400)

relu_values = np.maximum(0, z_values)
sigmoid_values = 1 / (1 + np.exp(-z_values))

plt.figure(figsize=(8, 5))
plt.plot(z_values, relu_values)
plt.axhline(0, linewidth=0.8)
plt.axvline(0, linewidth=0.8)
plt.xlabel("z")
plt.ylabel("ReLU(z)")
plt.title("Función ReLU")
plt.grid(alpha=0.2)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(z_values, sigmoid_values)
plt.axhline(0.5, linestyle="--", linewidth=0.8)
plt.axvline(0, linewidth=0.8)
plt.xlabel("z")
plt.ylabel("sigmoid(z)")
plt.title("Función Sigmoid")
plt.grid(alpha=0.2)
plt.show()


## Aplicar ReLU a nuestra neurona


In [ ]:
a_relu = max(0, z)

print("z =", z)
print("ReLU(z) =", a_relu)


# 4. De una neurona a una capa

Una capa contiene varias neuronas.

Matemáticamente:

\[
\mathbf{z}
=
\mathbf{W}\mathbf{x}
+
\mathbf{b}
\]

y después:

\[
\mathbf{a}=f(\mathbf{z})
\]

Una capa con 16 neuronas produce un vector de 16 salidas.


# 5. ¿Qué es una MLP?

Una **Multilayer Perceptron (MLP)** es una red formada por capas densas.

Nuestra arquitectura será:

**2 entradas**

↓

**Dense 16 + ReLU**

↓

**Dense 8 + ReLU**

↓

**Dense 2**

↓

**predicción**

Cada neurona de una capa densa recibe información de todas las salidas de la capa anterior.


# 6. Forward pass

El **forward pass** es el recorrido de la información desde la entrada hasta la salida.

\[
\mathbf{x}
\rightarrow
capa_1
\rightarrow
ReLU
\rightarrow
capa_2
\rightarrow
ReLU
\rightarrow
capa_{salida}
\rightarrow
logits
\]

Los **logits** son los valores crudos producidos por la última capa antes de convertirlos en probabilidades.


# 7. Softmax

Para logits:

\[
z_1,z_2,\ldots,z_C
\]

softmax produce:

\[
p_i =
\frac{e^{z_i}}
{\sum_j e^{z_j}}
\]

Las probabilidades:

- están entre 0 y 1;
- suman 1.

La clase predicha será la de mayor probabilidad.


In [ ]:
logits_demo = np.array([2.1, 0.3])

exp_values = np.exp(
    logits_demo - logits_demo.max()
)

probabilidades_demo = (
    exp_values / exp_values.sum()
)

print("Logits:", logits_demo)
print("Probabilidades:", probabilidades_demo)
print("Suma:", probabilidades_demo.sum())


# 8. ¿Cómo sabe la red si se equivoca?

Necesitamos una función de pérdida o **loss**.

Durante el entrenamiento queremos:

\[
\text{minimizar loss}
\]

En clasificación utilizaremos **cross-entropy**.

Intuitivamente:

- alta probabilidad para la clase correcta → menor pérdida;
- baja probabilidad para la clase correcta → mayor pérdida.

## Loss ≠ métrica

La loss se usa para optimizar.

Una métrica como accuracy se usa para interpretar el rendimiento.


# 9. Backpropagation

Tenemos:

1. pesos;
2. predicción;
3. loss.

Ahora necesitamos saber cómo cambiar los pesos.

Los **gradientes** indican cómo cambia la loss cuando modificamos un parámetro.

De forma simplificada:

\[
w_{\text{nuevo}}
=
w_{\text{actual}}
-
\eta
\frac{\partial L}{\partial w}
\]

Backpropagation aplica la regla de la cadena para calcular eficientemente los gradientes desde la salida hacia las capas anteriores.


## Ciclo completo

### Forward

\[
X \rightarrow red \rightarrow predicción
\]

### Loss

\[
predicción + respuesta\ real \rightarrow error
\]

### Backward

\[
error \rightarrow gradientes
\]

### Optimizer

\[
gradientes \rightarrow actualización\ de\ pesos
\]


# 10. Optimizer y learning rate

El optimizador decide cómo utilizar los gradientes para actualizar parámetros.

Ejemplos:

- SGD;
- Adam;
- RMSprop.

Usaremos **Adam**.

El **learning rate** controla el tamaño de los pasos.

Muy grande:

- entrenamiento inestable.

Muy pequeño:

- aprendizaje lento.


# 11. Batch, step y epoch

## Batch

Grupo de ejemplos procesados juntos.

```text
batch_size = 32
```

## Step / iteration

Una actualización de parámetros usando un batch.

## Epoch

Una pasada completa por todo train.

Si tenemos 800 ejemplos y `batch_size=32`:

\[
800/32 \approx 25
\]

steps por epoch.


# 12. Dataset: dos lunas

Usaremos un problema binario no lineal.

La ventaja es que podremos dibujar exactamente qué aprende la red.


In [ ]:
X, y = make_moons(
    n_samples=1200,
    noise=0.22,
    random_state=SEED
)

print("X:", X.shape)
print("y:", y.shape)

plt.figure(figsize=(8, 6))
plt.scatter(
    X[:, 0],
    X[:, 1],
    c=y,
    s=35,
    alpha=0.75
)
plt.xlabel("x1")
plt.ylabel("x2")
plt.title("Dataset: dos lunas")
plt.grid(alpha=0.2)
plt.show()


La frontera entre clases no puede representarse bien mediante una única recta.

Esto nos permitirá ver por qué la no linealidad es útil.


# 13. Train, validation y test

## Train
Actualiza pesos.

## Validation
Ayuda a tomar decisiones durante el desarrollo.

## Test
Se reserva para la evaluación final.


In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=SEED,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=SEED,
    stratify=y_temp
)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)


# 14. Escalar las entradas

Las redes suelen entrenar mejor con entradas en escalas razonables.

Ajustaremos `StandardScaler` **solo con train**.


In [ ]:
scaler = StandardScaler()

X_train_s = scaler.fit_transform(X_train)
X_val_s = scaler.transform(X_val)
X_test_s = scaler.transform(X_test)

print(
    "Media train:",
    X_train_s.mean(axis=0)
)

print(
    "Desviación train:",
    X_train_s.std(axis=0)
)


# PARTE I — KERAS

Keras permite definir y entrenar una red con muy poco código.

Primero veremos el flujo de alto nivel.


# 15. Construir la red

Arquitectura:

\[
2 \rightarrow 16 \rightarrow 8 \rightarrow 2
\]

La última capa devuelve **2 logits**.


In [ ]:
tf.random.set_seed(SEED)

modelo_keras = keras.Sequential([
    keras.Input(shape=(2,)),
    layers.Dense(
        16,
        activation="relu"
    ),
    layers.Dense(
        8,
        activation="relu"
    ),
    layers.Dense(2)
])

modelo_keras.summary()


# 16. ¿Cuántos parámetros hay?

En una capa Dense:

\[
n_{\text{parámetros}}
=
n_{\text{entradas}}
\times
n_{\text{neuronas}}
+
n_{\text{neuronas}}
\]

Primera capa:

\[
2\times16+16=48
\]

Segunda:

\[
16\times8+8=136
\]

Salida:

\[
8\times2+2=18
\]

Total:

\[
202
\]


# 17. Compilar

Usaremos:

- Adam;
- Sparse Categorical Crossentropy;
- logits;
- accuracy.


In [ ]:
modelo_keras.compile(
    optimizer=keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss=keras.losses.SparseCategoricalCrossentropy(
        from_logits=True
    ),
    metrics=["accuracy"]
)


# 18. Entrenar con `fit()`

Keras realizará internamente:

**batch → forward → loss → backward → actualización**


In [ ]:
history = modelo_keras.fit(
    X_train_s,
    y_train,
    validation_data=(
        X_val_s,
        y_val
    ),
    epochs=60,
    batch_size=32,
    verbose=0
)

print("Entrenamiento terminado.")


# 19. Curvas de aprendizaje

In [ ]:
hist_df = pd.DataFrame(
    history.history
)

hist_df.head()


In [ ]:
plt.figure(figsize=(9, 6))
plt.plot(hist_df["loss"], label="Train")
plt.plot(hist_df["val_loss"], label="Validation")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Keras — loss")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

plt.figure(figsize=(9, 6))
plt.plot(hist_df["accuracy"], label="Train")
plt.plot(hist_df["val_accuracy"], label="Validation")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Keras — accuracy")
plt.legend()
plt.grid(alpha=0.2)
plt.show()


## Interpretación

### Buen ajuste
Train y validation mejoran y permanecen relativamente próximos.

### Posible overfitting
Train sigue mejorando mientras validation deja de mejorar o empeora.

### Underfitting
Ambos permanecen con rendimiento bajo.


# 20. Evaluar en test

In [ ]:
test_loss_keras, test_acc_keras = modelo_keras.evaluate(
    X_test_s,
    y_test,
    verbose=0
)

print("Test loss:", round(test_loss_keras, 4))
print("Test accuracy:", round(test_acc_keras, 4))


# 21. Logits, probabilidades y predicciones

In [ ]:
logits_keras = modelo_keras.predict(
    X_test_s,
    verbose=0
)

prob_keras = tf.nn.softmax(
    logits_keras,
    axis=1
).numpy()

pred_keras = np.argmax(
    prob_keras,
    axis=1
)

print("Logits:")
print(logits_keras[:5])

print("\nProbabilidades:")
print(prob_keras[:5])

print("\nPredicciones:")
print(pred_keras[:5])


# 22. Frontera de decisión — Keras

In [ ]:
def frontera_keras(
    modelo,
    scaler,
    X_original,
    y,
    titulo,
    paso=0.02
):
    x_min = X_original[:, 0].min() - 0.6
    x_max = X_original[:, 0].max() + 0.6
    y_min = X_original[:, 1].min() - 0.6
    y_max = X_original[:, 1].max() + 0.6

    xx, yy = np.meshgrid(
        np.arange(x_min, x_max, paso),
        np.arange(y_min, y_max, paso)
    )

    rejilla = np.c_[
        xx.ravel(),
        yy.ravel()
    ]

    rejilla_s = scaler.transform(
        rejilla
    )

    logits = modelo.predict(
        rejilla_s,
        verbose=0
    )

    pred = np.argmax(
        logits,
        axis=1
    ).reshape(xx.shape)

    plt.figure(figsize=(8, 6))
    plt.contourf(
        xx,
        yy,
        pred,
        alpha=0.25
    )
    plt.scatter(
        X_original[:, 0],
        X_original[:, 1],
        c=y,
        s=35,
        alpha=0.7
    )
    plt.xlabel("x1")
    plt.ylabel("x2")
    plt.title(titulo)
    plt.show()


frontera_keras(
    modelo_keras,
    scaler,
    X_train,
    y_train,
    "Keras — frontera de decisión"
)


La MLP puede construir una frontera no lineal gracias a las capas ocultas y sus activaciones.


# 23. Matriz de confusión — Keras

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test,
    pred_keras
)

plt.title("Keras — matriz de confusión")
plt.show()

print(
    classification_report(
        y_test,
        pred_keras
    )
)


# PARTE II — PYTORCH

Ahora implementaremos una red equivalente.

La diferencia pedagógica es importante:

> En PyTorch escribiremos explícitamente el ciclo de entrenamiento.

Las líneas esenciales serán:

```python
optimizer.zero_grad()
logits = model(X)
loss = criterion(logits, y)
loss.backward()
optimizer.step()
```


# 24. Tensores

PyTorch utiliza tensores.

Son estructuras multidimensionales preparadas para:

- cálculo automático de gradientes;
- CPU/GPU;
- Deep Learning.


In [ ]:
X_train_t = torch.tensor(
    X_train_s,
    dtype=torch.float32
)

y_train_t = torch.tensor(
    y_train,
    dtype=torch.long
)

X_val_t = torch.tensor(
    X_val_s,
    dtype=torch.float32
)

y_val_t = torch.tensor(
    y_val,
    dtype=torch.long
)

X_test_t = torch.tensor(
    X_test_s,
    dtype=torch.float32
)

y_test_t = torch.tensor(
    y_test,
    dtype=torch.long
)

print(X_train_t.shape)
print(y_train_t.shape)


# 25. Dataset y DataLoader

`TensorDataset` une entradas y etiquetas.

`DataLoader` crea batches.

Con `shuffle=True`, cambia el orden de train entre epochs.


In [ ]:
BATCH_SIZE = 32

train_dataset = TensorDataset(
    X_train_t,
    y_train_t
)

val_dataset = TensorDataset(
    X_val_t,
    y_val_t
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

batch_X, batch_y = next(
    iter(train_loader)
)

print("Batch X:", batch_X.shape)
print("Batch y:", batch_y.shape)


# 26. Definir la red en PyTorch

Misma arquitectura:

\[
2 \rightarrow 16 \rightarrow 8 \rightarrow 2
\]


In [ ]:
class PrimeraRed(nn.Module):

    def __init__(self):
        super().__init__()

        self.red = nn.Sequential(
            nn.Linear(2, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, 2)
        )

    def forward(self, x):
        return self.red(x)


torch.manual_seed(SEED)

modelo_torch = PrimeraRed().to(
    device
)

print(modelo_torch)


# 27. Contar parámetros

In [ ]:
total_parametros = sum(
    p.numel()
    for p in modelo_torch.parameters()
)

print(
    "Parámetros totales:",
    total_parametros
)


Deberíamos obtener:

\[
202
\]

igual que en Keras.


# 28. Loss y optimizador

Usaremos:

```python
nn.CrossEntropyLoss()
```

Esta función recibe directamente:

- logits;
- etiquetas enteras.

Por eso no añadimos Softmax dentro de la red.


In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    modelo_torch.parameters(),
    lr=0.001
)


# 29. Training loop — aquí vemos realmente el aprendizaje

Para cada batch:

### 1. Borrar gradientes anteriores

```python
optimizer.zero_grad()
```

### 2. Forward

```python
logits = model(X)
```

### 3. Calcular loss

```python
loss = criterion(logits, y)
```

### 4. Backpropagation

```python
loss.backward()
```

### 5. Actualizar pesos

```python
optimizer.step()
```


In [ ]:
def evaluar_torch(
    modelo,
    loader,
    criterion,
    device
):
    modelo.eval()

    loss_total = 0.0
    correctos = 0
    total = 0

    with torch.no_grad():

        for X_batch, y_batch in loader:

            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            logits = modelo(X_batch)

            loss = criterion(
                logits,
                y_batch
            )

            loss_total += (
                loss.item()
                * X_batch.size(0)
            )

            pred = logits.argmax(
                dim=1
            )

            correctos += (
                pred == y_batch
            ).sum().item()

            total += y_batch.size(0)

    return (
        loss_total / total,
        correctos / total
    )


In [ ]:
EPOCHS = 60

history_torch = {
    "train_loss": [],
    "train_accuracy": [],
    "val_loss": [],
    "val_accuracy": []
}

for epoch in range(EPOCHS):

    modelo_torch.train()

    loss_total = 0.0
    correctos = 0
    total = 0

    for X_batch, y_batch in train_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        # 1. Borrar gradientes
        optimizer.zero_grad()

        # 2. Forward
        logits = modelo_torch(
            X_batch
        )

        # 3. Loss
        loss = criterion(
            logits,
            y_batch
        )

        # 4. Backpropagation
        loss.backward()

        # 5. Actualizar pesos
        optimizer.step()

        loss_total += (
            loss.item()
            * X_batch.size(0)
        )

        pred = logits.argmax(
            dim=1
        )

        correctos += (
            pred == y_batch
        ).sum().item()

        total += y_batch.size(0)

    train_loss = (
        loss_total / total
    )

    train_accuracy = (
        correctos / total
    )

    val_loss, val_accuracy = evaluar_torch(
        modelo_torch,
        val_loader,
        criterion,
        device
    )

    history_torch[
        "train_loss"
    ].append(train_loss)

    history_torch[
        "train_accuracy"
    ].append(train_accuracy)

    history_torch[
        "val_loss"
    ].append(val_loss)

    history_torch[
        "val_accuracy"
    ].append(val_accuracy)

print("Entrenamiento terminado.")


# 30. Curvas de aprendizaje — PyTorch

In [ ]:
hist_torch_df = pd.DataFrame(
    history_torch
)

plt.figure(figsize=(9, 6))
plt.plot(
    hist_torch_df["train_loss"],
    label="Train"
)
plt.plot(
    hist_torch_df["val_loss"],
    label="Validation"
)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("PyTorch — loss")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

plt.figure(figsize=(9, 6))
plt.plot(
    hist_torch_df["train_accuracy"],
    label="Train"
)
plt.plot(
    hist_torch_df["val_accuracy"],
    label="Validation"
)
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("PyTorch — accuracy")
plt.legend()
plt.grid(alpha=0.2)
plt.show()


# 31. Evaluación final — PyTorch

In [ ]:
test_dataset = TensorDataset(
    X_test_t,
    y_test_t
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)

test_loss_torch, test_acc_torch = evaluar_torch(
    modelo_torch,
    test_loader,
    criterion,
    device
)

print(
    "Test loss:",
    round(test_loss_torch, 4)
)

print(
    "Test accuracy:",
    round(test_acc_torch, 4)
)


# 32. Logits y probabilidades — PyTorch

In [ ]:
modelo_torch.eval()

with torch.no_grad():

    logits_torch = modelo_torch(
        X_test_t.to(device)
    )

    prob_torch = torch.softmax(
        logits_torch,
        dim=1
    ).cpu().numpy()

    pred_torch = logits_torch.argmax(
        dim=1
    ).cpu().numpy()

print("Logits:")
print(
    logits_torch[:5]
    .cpu()
    .numpy()
)

print("\nProbabilidades:")
print(prob_torch[:5])

print("\nPredicciones:")
print(pred_torch[:5])


# 33. Frontera de decisión — PyTorch

In [ ]:
def frontera_torch(
    modelo,
    scaler,
    X_original,
    y,
    device,
    titulo,
    paso=0.02
):
    x_min = X_original[:, 0].min() - 0.6
    x_max = X_original[:, 0].max() + 0.6
    y_min = X_original[:, 1].min() - 0.6
    y_max = X_original[:, 1].max() + 0.6

    xx, yy = np.meshgrid(
        np.arange(x_min, x_max, paso),
        np.arange(y_min, y_max, paso)
    )

    rejilla = np.c_[
        xx.ravel(),
        yy.ravel()
    ]

    rejilla_s = scaler.transform(
        rejilla
    )

    tensor_grid = torch.tensor(
        rejilla_s,
        dtype=torch.float32
    ).to(device)

    modelo.eval()

    with torch.no_grad():

        logits = modelo(
            tensor_grid
        )

        pred = logits.argmax(
            dim=1
        ).cpu().numpy()

    pred = pred.reshape(
        xx.shape
    )

    plt.figure(figsize=(8, 6))
    plt.contourf(
        xx,
        yy,
        pred,
        alpha=0.25
    )
    plt.scatter(
        X_original[:, 0],
        X_original[:, 1],
        c=y,
        s=35,
        alpha=0.7
    )
    plt.xlabel("x1")
    plt.ylabel("x2")
    plt.title(titulo)
    plt.show()


frontera_torch(
    modelo_torch,
    scaler,
    X_train,
    y_train,
    device,
    "PyTorch — frontera de decisión"
)


# 34. Matriz de confusión — PyTorch

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test,
    pred_torch
)

plt.title("PyTorch — matriz de confusión")
plt.show()

print(
    classification_report(
        y_test,
        pred_torch
    )
)


# 35. Keras vs. PyTorch

Hemos construido prácticamente la misma red.

## Keras

El entrenamiento se resume en:

```python
model.compile(...)
model.fit(...)
```

La librería gestiona internamente el ciclo.

## PyTorch

Escribimos explícitamente:

```python
zero_grad()
forward
loss
backward
step
```

## Conceptualmente hacen lo mismo

**datos → red → loss → gradientes → actualización**


In [ ]:
comparacion_frameworks = pd.DataFrame({
    "framework": [
        "Keras",
        "PyTorch"
    ],
    "test_accuracy": [
        test_acc_keras,
        test_acc_torch
    ],
    "test_loss": [
        test_loss_keras,
        test_loss_torch
    ]
})

comparacion_frameworks


No debemos interpretar pequeñas diferencias como que un framework sea “mejor”.

Los resultados pueden cambiar por:

- inicialización;
- orden de batches;
- detalles internos;
- precisión numérica;
- optimizador;
- semilla;
- hardware.

La elección del framework depende también de ergonomía, ecosistema y necesidades del proyecto.


# 36. ¿Qué pasa si quitamos la no linealidad?

Crearemos una red con varias capas, pero **sin ReLU**.

Aunque tenga varias capas lineales, la composición sigue siendo lineal.

Veremos la frontera.


In [ ]:
tf.random.set_seed(SEED)

modelo_lineal = keras.Sequential([
    keras.Input(shape=(2,)),
    layers.Dense(16),
    layers.Dense(8),
    layers.Dense(2)
])

modelo_lineal.compile(
    optimizer=keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss=keras.losses.SparseCategoricalCrossentropy(
        from_logits=True
    ),
    metrics=["accuracy"]
)

modelo_lineal.fit(
    X_train_s,
    y_train,
    validation_data=(
        X_val_s,
        y_val
    ),
    epochs=60,
    batch_size=32,
    verbose=0
)

frontera_keras(
    modelo_lineal,
    scaler,
    X_train,
    y_train,
    "Red sin activaciones no lineales"
)


Este experimento ayuda a entender una idea esencial:

> **añadir capas no basta; necesitamos no linealidad para representar relaciones no lineales complejas.**


# 37. Overfitting en redes neuronales

Una red con demasiada capacidad puede memorizar peculiaridades del train.

Señal típica:

- train loss sigue bajando;
- validation loss deja de bajar o sube.

Vamos a crear deliberadamente una red mucho más grande y entrenarla durante más epochs.


In [ ]:
tf.random.set_seed(SEED)

modelo_grande = keras.Sequential([
    keras.Input(shape=(2,)),
    layers.Dense(
        128,
        activation="relu"
    ),
    layers.Dense(
        128,
        activation="relu"
    ),
    layers.Dense(
        64,
        activation="relu"
    ),
    layers.Dense(2)
])

modelo_grande.compile(
    optimizer=keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss=keras.losses.SparseCategoricalCrossentropy(
        from_logits=True
    ),
    metrics=["accuracy"]
)

history_grande = modelo_grande.fit(
    X_train_s,
    y_train,
    validation_data=(
        X_val_s,
        y_val
    ),
    epochs=180,
    batch_size=32,
    verbose=0
)


In [ ]:
hist_grande = pd.DataFrame(
    history_grande.history
)

plt.figure(figsize=(9, 6))
plt.plot(
    hist_grande["loss"],
    label="Train"
)
plt.plot(
    hist_grande["val_loss"],
    label="Validation"
)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Red grande — train vs. validation")
plt.legend()
plt.grid(alpha=0.2)
plt.show()


Dependiendo de la ejecución, el sobreajuste puede ser más o menos marcado.

La idea importante es mirar **la separación entre train y validation**, no solo el rendimiento final de train.


# 38. Regularización

Algunas estrategias habituales:

- reducir la capacidad del modelo;
- obtener más datos;
- data augmentation, cuando tiene sentido;
- regularización L1/L2;
- Dropout;
- Early Stopping.

No todas son apropiadas para todos los problemas.


## Dropout

Durante el entrenamiento, Dropout anula aleatoriamente una fracción de activaciones.

Esto dificulta que la red dependa demasiado de combinaciones concretas de neuronas.

En inferencia, Dropout se desactiva.


In [ ]:
tf.random.set_seed(SEED)

modelo_dropout = keras.Sequential([
    keras.Input(shape=(2,)),
    layers.Dense(
        64,
        activation="relu"
    ),
    layers.Dropout(0.25),
    layers.Dense(
        32,
        activation="relu"
    ),
    layers.Dropout(0.25),
    layers.Dense(2)
])

modelo_dropout.compile(
    optimizer=keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss=keras.losses.SparseCategoricalCrossentropy(
        from_logits=True
    ),
    metrics=["accuracy"]
)

history_dropout = modelo_dropout.fit(
    X_train_s,
    y_train,
    validation_data=(
        X_val_s,
        y_val
    ),
    epochs=120,
    batch_size=32,
    verbose=0
)


## Early Stopping

Podemos detener el entrenamiento si validation loss deja de mejorar.


In [ ]:
tf.random.set_seed(SEED)

modelo_early = keras.Sequential([
    keras.Input(shape=(2,)),
    layers.Dense(
        64,
        activation="relu"
    ),
    layers.Dense(
        32,
        activation="relu"
    ),
    layers.Dense(2)
])

modelo_early.compile(
    optimizer=keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss=keras.losses.SparseCategoricalCrossentropy(
        from_logits=True
    ),
    metrics=["accuracy"]
)

early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=15,
    restore_best_weights=True
)

history_early = modelo_early.fit(
    X_train_s,
    y_train,
    validation_data=(
        X_val_s,
        y_val
    ),
    epochs=300,
    batch_size=32,
    callbacks=[early_stopping],
    verbose=0
)

print(
    "Epochs ejecutados:",
    len(history_early.history["loss"])
)


# 39. Experimento: learning rate

Compararemos varios valores.

Queremos observar cómo cambia la velocidad y estabilidad del aprendizaje.


In [ ]:
learning_rates = [
    0.0001,
    0.001,
    0.05
]

historias_lr = {}

for lr in learning_rates:

    tf.random.set_seed(SEED)

    modelo = keras.Sequential([
        keras.Input(shape=(2,)),
        layers.Dense(
            16,
            activation="relu"
        ),
        layers.Dense(
            8,
            activation="relu"
        ),
        layers.Dense(2)
    ])

    modelo.compile(
        optimizer=keras.optimizers.Adam(
            learning_rate=lr
        ),
        loss=keras.losses.SparseCategoricalCrossentropy(
            from_logits=True
        ),
        metrics=["accuracy"]
    )

    h = modelo.fit(
        X_train_s,
        y_train,
        validation_data=(
            X_val_s,
            y_val
        ),
        epochs=40,
        batch_size=32,
        verbose=0
    )

    historias_lr[lr] = h.history[
        "val_loss"
    ]


In [ ]:
plt.figure(figsize=(9, 6))

for lr, valores in historias_lr.items():
    plt.plot(
        valores,
        label=f"lr={lr}"
    )

plt.xlabel("Epoch")
plt.ylabel("Validation loss")
plt.title("Efecto del learning rate")
plt.legend()
plt.grid(alpha=0.2)
plt.show()


Un learning rate mayor no significa automáticamente aprender mejor o más rápido.

Puede avanzar más deprisa, pero también saltarse regiones útiles o volverse inestable.


# 40. Experimento: batch size

Compararemos:

```text
8
32
128
```

Batch pequeño:

- más actualizaciones por epoch;
- gradientes más ruidosos;
- menor memoria por batch.

Batch grande:

- menos actualizaciones por epoch;
- estimaciones de gradiente más estables;
- mayor memoria.

No existe un batch size universalmente mejor.


In [ ]:
batch_sizes = [
    8,
    32,
    128
]

resultados_batch = []

for batch_size in batch_sizes:

    tf.random.set_seed(SEED)

    modelo = keras.Sequential([
        keras.Input(shape=(2,)),
        layers.Dense(
            16,
            activation="relu"
        ),
        layers.Dense(
            8,
            activation="relu"
        ),
        layers.Dense(2)
    ])

    modelo.compile(
        optimizer=keras.optimizers.Adam(
            learning_rate=0.001
        ),
        loss=keras.losses.SparseCategoricalCrossentropy(
            from_logits=True
        ),
        metrics=["accuracy"]
    )

    h = modelo.fit(
        X_train_s,
        y_train,
        validation_data=(
            X_val_s,
            y_val
        ),
        epochs=50,
        batch_size=batch_size,
        verbose=0
    )

    resultados_batch.append({
        "batch_size": batch_size,
        "val_accuracy_final": h.history[
            "val_accuracy"
        ][-1],
        "val_loss_final": h.history[
            "val_loss"
        ][-1]
    })

pd.DataFrame(
    resultados_batch
)


# 41. Seguir una predicción individual

Elegiremos una muestra de test y observaremos:

1. valores originales;
2. valores escalados;
3. logits;
4. probabilidades;
5. clase final.


In [ ]:
indice = 0

muestra_original = X_test[
    indice:indice + 1
]

muestra_escalada = scaler.transform(
    muestra_original
)

logits = modelo_keras.predict(
    muestra_escalada,
    verbose=0
)

prob = tf.nn.softmax(
    logits,
    axis=1
).numpy()

pred = np.argmax(
    prob,
    axis=1
)[0]

print("Entrada original:")
print(muestra_original)

print("\nEntrada escalada:")
print(muestra_escalada)

print("\nLogits:")
print(logits)

print("\nProbabilidades:")
print(prob)

print("\nClase real:")
print(y_test[indice])

print("\nClase predicha:")
print(pred)


# 42. Mirar dentro de la primera capa

Las capas ocultas transforman la representación de los datos.

Podemos construir un modelo auxiliar que devuelva la salida de la primera capa.


In [ ]:
modelo_activaciones = keras.Model(
    inputs=modelo_keras.inputs,
    outputs=modelo_keras.layers[0].output
)

activaciones = modelo_activaciones.predict(
    X_test_s[:5],
    verbose=0
)

print(
    "Forma de las activaciones:",
    activaciones.shape
)

print(
    "\nActivaciones de la primera muestra:"
)

print(
    np.round(
        activaciones[0],
        3
    )
)


La entrada original tenía:

\[
2
\]

características.

Después de la primera capa tenemos:

\[
16
\]

activaciones.

La red está construyendo una **representación interna** diferente del dato original.


# 43. Ver un gradiente en PyTorch

PyTorch nos permite inspeccionar directamente gradientes.

Tomaremos un único batch y haremos forward + backward sin actualizar todavía.


In [ ]:
modelo_grad = PrimeraRed().to(
    device
)

criterion_grad = nn.CrossEntropyLoss()

X_batch, y_batch = next(
    iter(train_loader)
)

X_batch = X_batch.to(device)
y_batch = y_batch.to(device)

logits = modelo_grad(
    X_batch
)

loss = criterion_grad(
    logits,
    y_batch
)

loss.backward()

primera_capa = modelo_grad.red[0]

print(
    "Shape de los pesos:",
    primera_capa.weight.shape
)

print(
    "Shape de los gradientes:",
    primera_capa.weight.grad.shape
)

print(
    "\nPrimeros gradientes:"
)

print(
    primera_capa.weight.grad[:3]
)


Esto es backpropagation materializado:

> cada parámetro entrenable recibe un gradiente.

El optimizador utilizará esos valores para actualizar los parámetros.


# 44. `train()` y `eval()` en PyTorch

En PyTorch solemos alternar:

```python
model.train()
```

durante entrenamiento,

y:

```python
model.eval()
```

durante validación o inferencia.

Esto importa especialmente cuando existen capas cuyo comportamiento cambia entre entrenamiento e inferencia, como:

- Dropout;
- Batch Normalization.

También usamos:

```python
torch.no_grad()
```

cuando no necesitamos gradientes, reduciendo trabajo innecesario.


# 45. Tabla mental de conceptos

| Concepto | Qué significa |
|---|---|
| Peso | Parámetro que multiplica una entrada |
| Bias | Parámetro de desplazamiento |
| Activación | Introduce transformación, frecuentemente no lineal |
| Capa | Conjunto de neuronas |
| Forward pass | De entrada a predicción |
| Logit | Salida cruda antes de convertir a probabilidad |
| Loss | Error que optimizamos |
| Gradiente | Cómo cambia la loss respecto a un parámetro |
| Backpropagation | Cálculo eficiente de gradientes hacia atrás |
| Optimizer | Actualiza los parámetros |
| Learning rate | Tamaño de los pasos de actualización |
| Batch | Grupo de ejemplos procesados juntos |
| Step | Una actualización |
| Epoch | Una pasada por train |
| Validation | Datos para seguimiento/selección |
| Test | Evaluación final |
| Overfitting | Modelo demasiado adaptado a train |
| Dropout | Regularización mediante apagado aleatorio de activaciones en train |
| Early stopping | Detener cuando validation deja de mejorar |


# 46. El mismo entrenamiento visto de dos formas

## Keras

```python
model.compile(...)
model.fit(...)
```

Esconde muchos detalles para hacer el entrenamiento más cómodo.

## PyTorch

```python
for X, y in loader:
    optimizer.zero_grad()
    logits = model(X)
    loss = criterion(logits, y)
    loss.backward()
    optimizer.step()
```

Hace explícito el ciclo.

## Ninguno cambia la idea matemática

\[
X
\rightarrow
forward
\rightarrow
loss
\rightarrow
backpropagation
\rightarrow
optimizer
\rightarrow
pesos\ nuevos
\]


# 47. Errores frecuentes al empezar con redes

### 1. Pensar que más capas siempre es mejor
Una arquitectura mayor puede sobreajustar y ser innecesaria.

### 2. Evaluar únicamente train
Un train accuracy altísimo puede ocultar mala generalización.

### 3. Confundir logits con probabilidades
Los logits no tienen por qué sumar 1 ni estar entre 0 y 1.

### 4. Aplicar Softmax dos veces
Muchas losses reciben logits directamente.

### 5. Olvidar escalar datos numéricos
Puede dificultar la optimización.

### 6. Elegir learning rate al azar y no observar curvas
El entrenamiento debe diagnosticarse.

### 7. Usar test durante todo el desarrollo
Test debe mantenerse lo más independiente posible.

### 8. Creer que Keras “no hace backpropagation”
Sí lo hace; simplemente lo gestiona internamente.

### 9. Olvidar `optimizer.zero_grad()` en PyTorch
Los gradientes se acumulan por defecto.

### 10. Pensar que un framework define el algoritmo
La red y el proceso de optimización son conceptos independientes de Keras o PyTorch.


# 48. Mini reto final

Utiliza el dataset `make_moons`.

## Parte A — Keras

1. crea una red `2 → 32 → 16 → 2`;
2. utiliza ReLU;
3. Adam;
4. `learning_rate=0.001`;
5. entrena 80 epochs;
6. muestra curvas de loss y accuracy;
7. evalúa test;
8. representa la frontera;
9. muestra matriz de confusión.

## Parte B — Modificación

Prueba una sola modificación:

- `learning_rate`;
- `batch_size`;
- número de neuronas;
- Dropout.

Compara con el modelo inicial.

## Parte C — PyTorch

Construye una red equivalente e identifica explícitamente estas cinco líneas:

```python
zero_grad
forward
loss
backward
step
```

## Parte D — Explicación

En 10–15 líneas explica con tus palabras:

> **¿Cómo aprende realmente una red neuronal?**


In [ ]:
# TU CÓDIGO AQUÍ



# 49. Preguntas de comprensión

1. ¿Qué es un peso?
2. ¿Qué es un bias?
3. ¿Qué calcula una neurona antes de la activación?
4. ¿Para qué sirve ReLU?
5. ¿Por qué una red formada únicamente por capas lineales tiene limitaciones?
6. ¿Qué es un forward pass?
7. ¿Qué es un logit?
8. ¿Qué hace Softmax?
9. ¿Qué diferencia existe entre loss y accuracy?
10. ¿Qué es un gradiente?
11. ¿Qué hace backpropagation?
12. ¿Qué hace un optimizador?
13. ¿Qué controla el learning rate?
14. ¿Qué es un batch?
15. ¿Qué es un step?
16. ¿Qué es un epoch?
17. ¿Para qué sirve validation?
18. ¿Por qué test debe reservarse?
19. ¿Qué es overfitting?
20. ¿Qué señal de overfitting puede aparecer en las curvas?
21. ¿Qué hace Dropout?
22. ¿Qué hace Early Stopping?
23. ¿Por qué CrossEntropyLoss puede recibir logits directamente?
24. ¿Por qué PyTorch usa `zero_grad()`?
25. ¿Qué diferencia práctica existe entre `train()` y `eval()`?
26. ¿Por qué Keras necesita menos código?
27. ¿Por qué eso no significa que Keras haga menos operaciones?
28. ¿Qué significa inferencia?
29. ¿Qué representa una capa oculta?
30. ¿Qué has aprendido que sea independiente del framework?


# 50. Qué debes recordar

Una red neuronal aprende parámetros.

El ciclo fundamental es:

**datos**

↓

**forward pass**

↓

**predicción**

↓

**loss**

↓

**backpropagation**

↓

**gradientes**

↓

**optimizer**

↓

**nuevos pesos**

↓

**repetir**

Los términos clave:

- **neuronas** realizan transformaciones;
- **pesos y biases** son parámetros aprendidos;
- **activaciones** permiten representar relaciones no lineales;
- **loss** indica qué queremos minimizar;
- **gradientes** indican cómo cambiar parámetros;
- **backpropagation** calcula esos gradientes;
- **optimizer** actualiza los parámetros;
- **batch** define cuántos ejemplos procesamos juntos;
- **epoch** es una pasada completa por train;
- **learning rate** controla el tamaño de los pasos;
- **validation** ayuda a detectar problemas de generalización;
- **test** evalúa el resultado final.

## La idea más importante

No memorices:

```python
model.fit()
```

ni:

```python
loss.backward()
```

Entiende qué representan:

> **una red aprende ajustando parámetros para reducir una función de pérdida a partir de ejemplos.**

Keras y PyTorch son dos formas distintas de expresar el mismo proceso.


# 51. Después de este notebook

Este notebook prepara el terreno para redes más avanzadas:

### MNIST
Clasificación de imágenes sencillas.

### CNN
Capas diseñadas para explotar estructura espacial.

### Transfer Learning
Reutilización de modelos preentrenados.

### RNN / LSTM / GRU
Modelado de secuencias.

### Transformers
Atención y representaciones contextuales.

La lógica de entrenamiento seguirá siendo reconocible:

**forward → loss → backward → update**.


# Referencias técnicas

El notebook sigue las APIs y conceptos actuales de:

- **Keras / TensorFlow** para modelos `Sequential`, capas `Dense`, `compile`, `fit`, `evaluate` y callbacks.
- **PyTorch** para `nn.Module`, `Dataset`, `DataLoader`, `CrossEntropyLoss`, optimizadores y training loops.

JAX se deja fuera deliberadamente de esta primera introducción para no mezclar la comprensión de redes neuronales con un paradigma de programación adicional.
